In [52]:
# Inference speed test com e sem KV Cache com modelo unsloth/Llama-3.2-1B-Instruct

import json
import time
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import boto3
import os


def load_test_examples(path, n=10):
    examples = []
    with open(path, "r") as f:
        for _ in range(n):
            line = f.readline()
            if not line:
                break
            examples.append(json.loads(line))
    return examples

def parse_messages(text):
    """
    Extrai as mensagens do texto formatado do tipo:
    <|start_header_id|>system<|end_header_id|> ... <|eot_id|>
    <|start_header_id|>user<|end_header_id|> ... <|eot_id|>
    <|start_header_id|>assistant<|end_header_id|> ... <|eot_id|>
    
    Retorna só system e user numa lista para inferência.
    """
    messages = []
    parts = text.split("<|start_header_id|>")
    for part in parts:
        if not part.strip():
            continue
        role_end_idx = part.find("<|end_header_id|>")
        role = part[:role_end_idx].strip()
        content_start = role_end_idx + len("<|end_header_id|>")
        content_end = part.find("<|eot_id|>")
        content = part[content_start:content_end].strip()
        if role in ("system", "user"):
            messages.append({"role": role, "content": content})
    return messages

def format_for_model(messages):
    """
    Junta system e user numa string para o modelo LLaMA.
    """
    formatted = ""
    for msg in messages:
        formatted += f"<|start_header_id|>{msg['role']}<|end_header_id|>\n{msg['content']}\n<|eot_id|>\n"
    return formatted.strip()


time_cache_base_llama = []
time_no_cache_base_llama = []


def main1():
    global time_cache_base_llama, time_no_cache_base_llama
    model_name = "unsloth/Llama-3.2-1B-Instruct"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16).cuda()
    
    test_path = "test2.jsonl"
    examples = load_test_examples(test_path, n=10)
    

    for i, example in enumerate(examples):
        print(f"\n\n=== Example {i+1} ===")
        messages = parse_messages(example["text"])
        prompt = format_for_model(messages)

        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        
        # Inferência COM KV Cache
        start = time.time()
        output_ids_cache = model.generate(**inputs, max_new_tokens=128, use_cache=True)
        t_cache_base_llama = time.time() - start
        output_cache = tokenizer.decode(output_ids_cache[0], skip_special_tokens=True)
        time_cache_base_llama.append(t_cache_base_llama)
        
        # Inferência SEM KV Cache
        start = time.time()
        output_ids_no_cache = model.generate(**inputs, max_new_tokens=128, use_cache=False)
        t_no_cache_base_llama = time.time() - start
        output_no_cache = tokenizer.decode(output_ids_no_cache[0], skip_special_tokens=True)
        time_no_cache_base_llama.append(t_no_cache_base_llama)

        print(f"Com KV Cache: {t_cache_base_llama:.3f}s")
        print(f"Sem KV Cache: {t_no_cache_base_llama:.3f}s")

        # Opcional: imprimir só a parte nova gerada (após o prompt)
        generated_cache = output_cache[len(prompt):].strip()
        generated_no_cache = output_no_cache[len(prompt):].strip()

        print("Generated with cache:", generated_cache[:300], "...")
        print("Generated without cache:", generated_no_cache[:300], "...")
        

if __name__ == "__main__":
    main1()



=== Example 1 ===
Com KV Cache: 2.815s
Sem KV Cache: 10.577s
Generated with cache: r is expressing satisfaction with their car's performance and longevity, mentioning that the engine still pulls strong and the car is aging nicely.

**Predicted Rating:**
Based on the review, I predict a rating of **4.500**. The reviewer's positive comments about the car's performance, maintenance,  ...
Generated without cache: mentions that the car is "aging quite nicely" and plans to keep it for a few more years, indicating satisfaction with the car's condition.

**Predicted Rating:**
Based on the review, I predict a **Rating of 4.200**. The reviewer mentions several positive aspects of the car, such as its performance,  ...


=== Example 2 ===
Com KV Cache: 2.859s
Sem KV Cache: 10.551s
Generated with cache: s frustration and disappointment with their car's transmission failure, which they attribute to poor maintenance from Nissan's service department. They also express concern about the cost of a re

In [53]:
# Inference speed test com e sem KV Cache com modelo fine-tuned

os.environ["AWS_DEFAULT_REGION"] = "eu-west-1"

runtime = boto3.client("sagemaker-runtime")

ENDPOINT_NAME = "g5-llama32-ft-2025-07-28-18-15-12"
TEST_FILE = "test2.jsonl"

def load_test_examples(path, n=10):
    examples = []
    with open(path, "r") as f:
        for _ in range(n):
            line = f.readline()
            if not line:
                break
            examples.append(json.loads(line))
    return examples

def parse_messages(text):
    messages = []
    parts = text.split("<|start_header_id|>")
    for part in parts:
        if not part.strip():
            continue
        role_end_idx = part.find("<|end_header_id|>")
        role = part[:role_end_idx].strip()
        content_start = role_end_idx + len("<|end_header_id|>")
        content_end = part.find("<|eot_id|>")
        content = part[content_start:content_end].strip()
        if role in ("system", "user"):
            messages.append({"role": role, "content": content})
    return messages

def format_for_model(messages):
    formatted = ""
    for msg in messages:
        formatted += f"<|start_header_id|>{msg['role']}<|end_header_id|>\n{msg['content']}\n<|eot_id|>\n"
    return formatted.strip()

def query_endpoint(prompt, use_cache=True):
    payload = {
        "inputs": prompt,
        "use_cache": use_cache,
    }

    response = runtime.invoke_endpoint(
        EndpointName=ENDPOINT_NAME,
        ContentType="application/json",
        Body=json.dumps(payload)
    )
    result = response["Body"].read().decode("utf-8")
    return result


time_cache_ft = []
time_nocache_ft = []


def main2():
    global time_cache_ft, time_nocache_ft
    examples = load_test_examples(TEST_FILE, n=10)


    for i, example in enumerate(examples):
        print(f"\n--- Exemplo {i+1} ---")
        messages = parse_messages(example["text"])
        prompt = format_for_model(messages)

        # Com KV Cache
        start = time.time()
        out_cache_ft = query_endpoint(prompt, use_cache=True)
        t_cache_ft = time.time() - start
        time_cache_ft.append(t_cache_ft)

        # Sem KV Cache
        start = time.time()
        out_nocache_ft = query_endpoint(prompt, use_cache=False)
        t_nocache_ft = time.time() - start
        time_nocache_ft.append(t_nocache_ft)

        print(f"✅ Tempo com KV Cache: {t_cache_ft:.3f}s")
        print(f"🚫 Tempo sem KV Cache: {t_nocache_ft:.3f}s")

        print("\n🔹 Resultado com cache:", out_cache_ft.strip()[:300])
        print("\n🔸 Resultado sem cache:", out_nocache_ft.strip()[:300])


if __name__ == "__main__":
    main2()


--- Exemplo 1 ---
✅ Tempo com KV Cache: 0.303s
🚫 Tempo sem KV Cache: 0.250s

🔹 Resultado com cache: {"generated_text": "<|start_header_id|>assistant<|end_header_id|>\n\n**Sentiment**: Positive\n**Rating**: 4.625\n**Response**: [To be generated during inference, if necessary"}

🔸 Resultado sem cache: {"generated_text": "<|start_header_id|>assistant<|end_header_id|>\n\n**Sentiment**: Positive\n**Rating**: 4.625\n**Response**: [To be generated during inference, if necessary"}

--- Exemplo 2 ---
✅ Tempo com KV Cache: 0.250s
🚫 Tempo sem KV Cache: 0.249s

🔹 Resultado com cache: {"generated_text": "<|start_header_id|>assistant<|end_header_id|>\n\n**Sentiment**: Positive\n**Rating**: 4.250\n**Response**: [To be generated during inference, if necessary"}

🔸 Resultado sem cache: {"generated_text": "<|start_header_id|>assistant<|end_header_id|>\n\n**Sentiment**: Positive\n**Rating**: 4.250\n**Response**: [To be generated during inference, if necessary"}

--- Exemplo 3 ---
✅ Tempo com KV Cache: 0

In [54]:
# Inference speed test com e sem KV Cache com modelo fine-tuned e prompt grande

# ⚙️ Nome do endpoint SageMaker
ENDPOINT_NAME = "g5-llama32-ft-2025-07-28-18-15-12"  # substitui pelo nome real

# Prompt longo (pelo menos 300 tokens)
long_prompt = """
A indústria da moda é uma das mais poluentes do mundo. A produção em massa de roupas, muitas vezes em condições de trabalho precárias, contribui significativamente para a poluição da água, emissões de carbono e geração de resíduos. Têxteis sintéticos, como o poliéster, liberam microplásticos nos oceanos. 
Além disso, a moda rápida incentiva o consumo excessivo, levando a enormes quantidades de roupas descartadas anualmente em aterros sanitários. 
A produção de algodão também demanda grandes quantidades de água e pesticidas, afetando ecossistemas e a saúde humana. 
A cadeia de suprimentos da moda global envolve transporte internacional que contribui para a pegada de carbono. 
Portanto, há uma crescente demanda por práticas sustentáveis, como a reciclagem de tecidos, uso de materiais orgânicos e comércio justo. 
As empresas precisam assumir responsabilidades ambientais e sociais para minimizar os impactos negativos dessa indústria que, embora criativa e culturalmente relevante, tem um custo ambiental elevado.
""".strip()

# Inicializa o cliente do SageMaker
sagemaker_runtime = boto3.client("sagemaker-runtime")

def call_sagemaker(prompt: str, use_cache: bool):
    payload = {
        "inputs": prompt,
        "use_cache": use_cache,
        "max_new_tokens": 100,
        "temperature": 0.7,
    }

    start = time.time()
    response = sagemaker_runtime.invoke_endpoint(
        EndpointName=ENDPOINT_NAME,
        ContentType="application/json",
        Body=json.dumps(payload),
    )
    elapsed = time.time() - start

    response_body = response["Body"].read().decode("utf-8")

    try:
        result = json.loads(response_body)
    except Exception:
        result = {"raw_response": response_body}

    return result, elapsed

# ---------- Execução com cache ----------
print("\n--- Com KV Cache ---")
result_cache, time_cache = call_sagemaker(long_prompt, use_cache=True)
print(f"⏱ Tempo: {time_cache:.3f}s")
print("🔹 Resposta:", result_cache)

# ---------- Execução sem cache ----------
print("\n--- Sem KV Cache ---")
result_no_cache, time_no_cache = call_sagemaker(long_prompt, use_cache=False)
print(f"⏱ Tempo: {time_no_cache:.3f}s")
print("🔸 Resposta:", result_no_cache)

# ---------- Comparação ----------
diff = time_no_cache - time_cache
print(f"\n📊 Diferença de tempo: {diff:.3f}s ({'cache mais rápido' if diff > 0 else 'cache mais lento ou ignorado'})")


--- Com KV Cache ---
⏱ Tempo: 0.279s
🔹 Resposta: {'generated_text': ' \nA reciclagem de tecidos pode ser uma solução para reduzir a quantidade de roupas descartadas, mas também'}

--- Sem KV Cache ---
⏱ Tempo: 0.240s
🔸 Resposta: {'generated_text': ' \nA reciclagem de tecidos pode ser uma solução para reduzir a quantidade de roupas descartadas, mas também'}

📊 Diferença de tempo: -0.038s (cache mais lento ou ignorado)


In [55]:
# Inference speed test com e sem KV Cache o endpoint utilizado para zero e few shots (meta-textgenerationneuron-llama-3-2-1b-2025-07-11-20-51-32-569)

os.environ["AWS_DEFAULT_REGION"] = "eu-west-1"

runtime = boto3.client("sagemaker-runtime")

ENDPOINT_NAME = "meta-textgenerationneuron-llama-3-2-1b-2025-07-11-20-51-32-569"
TEST_FILE = "test2.jsonl"


def load_test_examples(path, n=10):
    examples = []
    with open(path, "r") as f:
        for _ in range(n):
            line = f.readline()
            if not line:
                break
            examples.append(json.loads(line))
    return examples

    
def parse_messages(text):
    messages = []
    parts = text.split("<|start_header_id|>")
    for part in parts:
        if not part.strip():
            continue
        role_end_idx = part.find("<|end_header_id|>")
        role = part[:role_end_idx].strip()
        content_start = role_end_idx + len("<|end_header_id|>")
        content_end = part.find("<|eot_id|>")
        content = part[content_start:content_end].strip()
        if role in ("system", "user"):
            messages.append({"role": role, "content": content})
    return messages


def format_for_model(messages):
    formatted = ""
    for msg in messages:
        formatted += f"<|start_header_id|>{msg['role']}<|end_header_id|>\n{msg['content']}\n<|eot_id|>\n"
    return formatted.strip()


def query_endpoint(prompt, use_cache=True):
    payload = {
        "inputs": prompt,
        "use_cache": use_cache,
    }

    response = runtime.invoke_endpoint(
        EndpointName=ENDPOINT_NAME,
        ContentType="application/json",
        Body=json.dumps(payload)
    )
    result = response["Body"].read().decode("utf-8")
    return result
    

times_cache = []
times_nocache = []


def main3():
    global times_cache, times_nocache  # declara que vamos usar as globais
    examples = load_test_examples(TEST_FILE, n=10)

    for i, example in enumerate(examples):
        print(f"\n--- Exemplo {i+1} ---")
        messages = parse_messages(example["text"])
        prompt = format_for_model(messages)

        # Com KV Cache
        start = time.time()
        out_cache = query_endpoint(prompt, use_cache=True)
        t_cache = time.time() - start
        times_cache.append(t_cache)
        
        # Sem KV Cache
        start = time.time()
        out_nocache = query_endpoint(prompt, use_cache=False)
        t_nocache = time.time() - start
        times_nocache.append(t_nocache)
            
        print(f"✅ Tempo com KV Cache: {t_cache:.3f}s")
        print(f"🚫 Tempo sem KV Cache: {t_nocache:.3f}s")

        print("\n🔹 Resultado com cache:", out_cache.strip()[:300])
        print("\n🔸 Resultado sem cache:", out_nocache.strip()[:300])

    # Cálculo das médias
    avg_cache_base_llama = sum(time_cache_base_llama) / len(time_cache_base_llama)
    avg_no_cache_base_llama = sum(time_no_cache_base_llama) / len(time_no_cache_base_llama)
    diff_base_llama = avg_cache_base_llama - avg_no_cache_base_llama

    avg_cache_ft = sum(time_cache_ft) / len(time_cache_ft)
    avg_nocache_ft = sum(time_nocache_ft) / len(time_nocache_ft)
    diff_ft = avg_cache_ft - avg_nocache_ft
    
    
    avg_cache = sum(times_cache) / len(times_cache)
    avg_nocache = sum(times_nocache) / len(times_nocache)
    diff = avg_cache - avg_nocache
    
    with open('Inference Speed.txt', "w", encoding="utf-8") as f_out:
        f_out.write(f"✅ Base Llama Tempo com KV Cache: {avg_cache_base_llama:.3f}s\n")
        f_out.write(f"🚫 Base Llama Tempo sem KV Cache: {avg_no_cache_base_llama:.3f}s\n")
        f_out.write(f" Base Llama Tempo sem KV Cache: {diff_base_llama:.3f}s\n\n")
        
        f_out.write(f"✅ Fine-tuning Tempo com KV Cache: {avg_cache_ft:.3f}s\n")
        f_out.write(f"🚫 Fine-tuning Tempo sem KV Cache: {avg_nocache_ft:.3f}s\n")
        f_out.write(f" Fine-tuning Tempo sem KV Cache: {diff_ft:.3f}s\n\n")
        
        f_out.write(f"✅ Endpoint meta-llama-3-2-1b Tempo com KV Cache: {avg_cache:.3f}s\n")
        f_out.write(f"🚫 Endpoint meta-llama-3-2-1b Tempo sem KV Cache: {avg_nocache:.3f}s\n")
        f_out.write(f" Endpoint meta-llama-3-2-1b Tempo sem KV Cache: {diff:.3f}s\n\n")
        

    print(f' Inference Speed.txt criado')

    # Enviar para S3
    s3_client = boto3.client('s3', region_name='eu-west-1')
    s3_client.upload_file('Inference Speed.txt', 'i32419', 'output/Inference Speed.txt')
    print('Inference Speed.txt enviado para S3')


if __name__ == "__main__":
    main3()


--- Exemplo 1 ---
✅ Tempo com KV Cache: 1.140s
🚫 Tempo sem KV Cache: 1.084s

🔹 Resultado com cache: {"generated_text": "\n<|begin_of_text|>definitely\nDefinitely\nDefinitely\nDefinitely\nDefinitely\nDefinitely\nDefinitely\nDefinitely\nDefinitely\nDef"}

🔸 Resultado sem cache: {"generated_text": "\n<|begin_of_text|>definitely\nDefinitely\nDefinitely\nDefinitely\nDefinitely\nDefinitely\nDefinitely\nDefinitely\nDefinitely\nDef"}

--- Exemplo 2 ---
✅ Tempo com KV Cache: 1.088s
🚫 Tempo sem KV Cache: 1.085s

🔹 Resultado com cache: {"generated_text": "\n<|begin_of_text|>definitely not a Nissan fan\nDefinitely not a Nissan fan\nDefinitely not a Nissan fan\nDefinitely not a Nissan fan\n"}

🔸 Resultado sem cache: {"generated_text": "\n<|begin_of_text|>definitely not a Nissan fan\nDefinitely not a Nissan fan\nDefinitely not a Nissan fan\nDefinitely not a Nissan fan\n"}

--- Exemplo 3 ---
✅ Tempo com KV Cache: 1.084s
🚫 Tempo sem KV Cache: 1.093s

🔹 Resultado com cache: {"generated_text": "\n<|beg